In [1]:
import pandas as pd
from pprint import pprint
#filter to only the rows where the model is resnet50
# gb = cc.groupby(["backbone", "pooling", "sampler", "weight_config", "fulltune"])
coralcam_groups = ['aggression', 'biting']
fishfollow_groups = ['habitat', 'movement', 'bites', 'social_interaction', 'not_visible']
cc = pd.read_csv("../results/random/coralcam_results_label_tolerance_7.csv")
ff = pd.read_csv("../results/random/fishfollow_results_label_tolerance_7.csv")

In [2]:
import re
def parse_weight_config(weight_config):
    s = str(weight_config)

    # Case 1: plain string (ex: weight_method='focal_loss')
    match1 = re.search(
        r"weight_method\s*[:=]\s*['\"]?([A-Za-z_]+)",
        s
    )

    # Case 2: dict-like (ex: {'weight_method': 'focal_loss', ...})
    match2 = re.search(
        r"'weight_method'\s*:\s*['\"]?([A-Za-z_]+)",
        s
    )

    if match1:
        return match1.group(1)
    if match2:
        return match2.group(1)
    return None

cc['ci_strategy'] = cc['weight_config'].apply(parse_weight_config)
ff['ci_strategy'] = ff['weight_config'].apply(parse_weight_config)

cc.fillna({"freeze_backbone": False}, inplace=True)
cc['fulltune_status'] = cc['fulltune'] & ~cc['freeze_backbone']

ff.fillna({"freeze_backbone": False}, inplace=True)
ff['fulltune_status'] = ff['fulltune'] & ~ff['freeze_backbone']

/tmp/ipykernel_164788/195439653.py:26: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  cc.fillna({"freeze_backbone": False}, inplace=True)
/tmp/ipykernel_164788/195439653.py:29: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ff.fillna({"freeze_backbone": False}, inplace=True)


In [3]:
def get_full_table(df: pd.DataFrame, group_names, max_metric):
    df = df[df['fulltune_status'] == False]
    best_rows = df.loc[df.groupby(["backbone", 'pooling', 'ci_strategy'])[max_metric].idxmax()]
    return best_rows[["backbone", "pooling", "ci_strategy"] + [max_metric] +
            [f"{group}_{max_metric}" for group in group_names] ]

In [4]:
get_full_table(cc, coralcam_groups, "f1_macro")

,backbone,pooling,ci_strategy,f1_macro,aggression_f1_macro,biting_f1_macro
3,resnet50,mean,uniform,0.172694,0.0000,0.259042
0,videomae,mean,focal_loss,0.075720,0.0000,0.113580
2,videomae,mean,inverse,0.087210,0.0525,0.104565
1,videomae,mean,uniform,0.069330,0.0000,0.103996


In [5]:
get_full_table(cc, coralcam_groups, "mAP")

,backbone,pooling,ci_strategy,mAP,aggression_mAP,biting_mAP
3,resnet50,mean,uniform,0.124029,0.000355,0.185866
0,videomae,mean,focal_loss,0.093653,0.009448,0.135756
2,videomae,mean,inverse,0.071898,0.015248,0.100222
1,videomae,mean,uniform,0.091865,0.008257,0.133670


In [6]:
get_full_table(ff, fishfollow_groups, "f1_macro")

,backbone,pooling,ci_strategy,f1_macro,habitat_f1_macro,movement_f1_macro,bites_f1_macro,social_interaction_f1_macro,not_visible_f1_macro
3,resnet50,mean,uniform,0.235542,0.425554,0.287900,0.000000,0.000000,0.345889
0,videomae,mean,focal_loss,0.281293,0.473496,0.383625,0.000000,0.000000,0.318193
2,videomae,mean,inverse,0.324611,0.443459,0.442466,0.059859,0.075721,0.421938
1,videomae,mean,uniform,0.279002,0.466433,0.381443,0.000000,0.000000,0.320514


In [7]:
get_full_table(ff, fishfollow_groups, "mAP")

,backbone,pooling,ci_strategy,mAP,habitat_mAP,movement_mAP,bites_mAP,social_interaction_mAP,not_visible_mAP
3,resnet50,mean,uniform,0.257332,0.415202,0.337200,0.002422,0.026185,0.380256
0,videomae,mean,focal_loss,0.316929,0.480822,0.434335,0.005067,0.062529,0.428201
2,videomae,mean,inverse,0.295956,0.448582,0.407475,0.004568,0.055047,0.395555
1,videomae,mean,uniform,0.304661,0.481545,0.392208,0.005145,0.113898,0.425589
